## Baseline training model

We built a simple baseline model using the standardized training files input_2023_w01_standardized.csv and output_2023_w01_standardized.csv. From the input file, we used the target players only (player_to_predict == 1) and extracted the most important features from their last few observed frames, including position (x, y), speed (s), acceleration (a), movement and body angles (dir, o, converted to sine and cosine), short-term motion changes, the ball landing location (ball_land_x, ball_land_y), the player’s relative distance to that landing point, and time-horizon information such as num_frames_output and the future step t. Instead of predicting the whole future trajectory at once, we converted the task into many small training samples, where the same history is used to predict each future frame separately. We trained two XGBoost regressors to predict the player’s future displacement, dx and dy, relative to the last observed position, and then converted those back into predicted future coordinates x and y. As outputs, we obtained validation predictions and RMSE-based evaluation metrics, and we saved the trained models (model_dx.joblib, model_dy.joblib), the feature metadata (metadata.json), the validation predictions (validation_predictions.csv), the validation metrics (validation_metrics.csv), and a demo prediction file (baseline_predictions_demo.csv).

This notebook is based on the Week 1 (w01) data read from my local Colab environment. The preprocessing input uses the file that has only been standardized to a unified coordinate system. In the same way, we can generate files for other weeks as well. Later, we can apply the prediction after the final preprocessing dataset, then input data from other weeks to obtain more training results, .

In [11]:
# If you are using Colab, uncomment this:
# !pip install xgboost joblib -q

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBRegressor

from google.colab import drive
import os

drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/DataScience')

print(os.getcwd())
print(os.listdir())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/DataScience
['Untitled0.ipynb', 'input_2023_w01.csv', 'output_2023_w01.csv', 'input_2023_w01_standardized.csv', 'output_2023_w01_standardized.csv', 'preprocessing.ipynb', 'input_2023_w01.gsheet', 'input_2023_w01_standardized.gsheet']


In [12]:
# =========================
# Config
# =========================
TRAIN_INPUT_PATH = "input_2023_w01_standardized.csv"
TRAIN_OUTPUT_PATH = "output_2023_w01_standardized.csv"

MODEL_DIR = Path("baseline_model")
LAST_K = 5
RANDOM_STATE = 42

In [13]:
# =========================
# Helper functions
# =========================
REQUIRED_INPUT_COLUMNS = {
    "game_id",
    "play_id",
    "nfl_id",
    "frame_id",
    "player_to_predict",
    "x",
    "y",
    "s",
    "a",
    "dir",
    "o",
    "num_frames_output",
    "ball_land_x",
    "ball_land_y",
}

REQUIRED_OUTPUT_COLUMNS = {"game_id", "play_id", "nfl_id", "frame_id", "x", "y"}


def check_columns(df: pd.DataFrame, required, name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")


def add_angle_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ["dir", "o"]:
        radians = np.deg2rad(out[col].astype(float))
        out[f"{col}_sin"] = np.sin(radians)
        out[f"{col}_cos"] = np.cos(radians)
    return out


def _pad_to_last_k(group: pd.DataFrame, last_k: int) -> pd.DataFrame:
    group = group.sort_values("frame_id").reset_index(drop=True)
    if len(group) >= last_k:
        return group.iloc[-last_k:].reset_index(drop=True)

    first_row = group.iloc[[0]].copy()
    pad_count = last_k - len(group)
    pad = pd.concat([first_row] * pad_count, ignore_index=True)
    return pd.concat([pad, group], ignore_index=True).reset_index(drop=True)


def _history_features(player_hist: pd.DataFrame, last_k: int):
    hist = _pad_to_last_k(player_hist, last_k)
    hist = hist.sort_values("frame_id").reset_index(drop=True)

    last_row = hist.iloc[-1]
    first_row = hist.iloc[0]

    feat = {
        "last_x": float(last_row["x"]),
        "last_y": float(last_row["y"]),
        "last_s": float(last_row["s"]),
        "last_a": float(last_row["a"]),
        "ball_land_x": float(last_row["ball_land_x"]),
        "ball_land_y": float(last_row["ball_land_y"]),
        "land_dx": float(last_row["ball_land_x"] - last_row["x"]),
        "land_dy": float(last_row["ball_land_y"] - last_row["y"]),
        "num_frames_output": float(last_row["num_frames_output"]),
        "mean_s_last_k": float(hist["s"].mean()),
        "mean_a_last_k": float(hist["a"].mean()),
        "delta_x_over_window": float(last_row["x"] - first_row["x"]),
        "delta_y_over_window": float(last_row["y"] - first_row["y"]),
    }

    if len(hist) >= 2:
        prev_row = hist.iloc[-2]
        feat["delta_x_last_step"] = float(last_row["x"] - prev_row["x"])
        feat["delta_y_last_step"] = float(last_row["y"] - prev_row["y"])
    else:
        feat["delta_x_last_step"] = 0.0
        feat["delta_y_last_step"] = 0.0

    if "absolute_yardline_number" in hist.columns:
        feat["absolute_yardline_number"] = float(last_row["absolute_yardline_number"])

    if "was_flipped" in hist.columns:
        feat["was_flipped"] = float(bool(last_row["was_flipped"]))

    for i, row in hist.iterrows():
        prefix = f"hist_{i+1}"
        feat[f"{prefix}_x_rel"] = float(row["x"] - last_row["x"])
        feat[f"{prefix}_y_rel"] = float(row["y"] - last_row["y"])
        feat[f"{prefix}_s"] = float(row["s"])
        feat[f"{prefix}_a"] = float(row["a"])
        feat[f"{prefix}_dir_sin"] = float(row["dir_sin"])
        feat[f"{prefix}_dir_cos"] = float(row["dir_cos"])
        feat[f"{prefix}_o_sin"] = float(row["o_sin"])
        feat[f"{prefix}_o_cos"] = float(row["o_cos"])

    return feat


def build_training_table(input_df: pd.DataFrame, output_df: pd.DataFrame, last_k: int = 5) -> pd.DataFrame:
    input_df = add_angle_features(input_df)
    input_df = input_df.sort_values(["game_id", "play_id", "nfl_id", "frame_id"]).copy()
    output_df = output_df.sort_values(["game_id", "play_id", "nfl_id", "frame_id"]).copy()

    target_input = input_df[input_df["player_to_predict"] == 1].copy()

    rows = []

    grouped_target_input = target_input.groupby(["game_id", "play_id", "nfl_id"], sort=False)
    grouped_output = output_df.groupby(["game_id", "play_id", "nfl_id"], sort=False)

    for key, player_hist in grouped_target_input:
        if key not in grouped_output.groups:
            continue

        future = grouped_output.get_group(key).sort_values("frame_id").reset_index(drop=True)
        feat = _history_features(player_hist, last_k=last_k)

        last_obs = player_hist.sort_values("frame_id").iloc[-1]
        last_x = float(last_obs["x"])
        last_y = float(last_obs["y"])

        game_id, play_id, nfl_id = key
        play_key = f"{game_id}_{play_id}"

        for _, future_row in future.iterrows():
            row = {
                "game_id": game_id,
                "play_id": play_id,
                "nfl_id": nfl_id,
                "play_key": play_key,
                "future_frame_id": int(future_row["frame_id"]),
                "horizon_t": int(future_row["frame_id"]),
                "horizon_frac": float(future_row["frame_id"]) / max(float(feat["num_frames_output"]), 1.0),
                "target_x": float(future_row["x"]),
                "target_y": float(future_row["y"]),
                "target_dx": float(future_row["x"] - last_x),
                "target_dy": float(future_row["y"] - last_y),
            }
            row.update(feat)
            rows.append(row)

    if not rows:
        raise ValueError("No training rows were built. Check that input/output keys match.")

    return pd.DataFrame(rows)


def build_prediction_table(input_df: pd.DataFrame, last_k: int = 5) -> pd.DataFrame:
    input_df = add_angle_features(input_df)
    input_df = input_df.sort_values(["game_id", "play_id", "nfl_id", "frame_id"]).copy()
    target_input = input_df[input_df["player_to_predict"] == 1].copy()

    rows = []
    grouped_target_input = target_input.groupby(["game_id", "play_id", "nfl_id"], sort=False)

    for key, player_hist in grouped_target_input:
        feat = _history_features(player_hist, last_k=last_k)
        game_id, play_id, nfl_id = key
        play_key = f"{game_id}_{play_id}"
        num_future = int(player_hist["num_frames_output"].iloc[-1])

        for t in range(1, num_future + 1):
            row = {
                "game_id": game_id,
                "play_id": play_id,
                "nfl_id": nfl_id,
                "play_key": play_key,
                "future_frame_id": t,
                "horizon_t": t,
                "horizon_frac": float(t) / max(float(feat["num_frames_output"]), 1.0),
            }
            row.update(feat)
            rows.append(row)

    if not rows:
        raise ValueError("No prediction rows were built. Check player_to_predict in input data.")

    return pd.DataFrame(rows)


def get_feature_columns(df: pd.DataFrame):
    ignore = {
        "game_id",
        "play_id",
        "nfl_id",
        "play_key",
        "future_frame_id",
        "target_x",
        "target_y",
        "target_dx",
        "target_dy",
    }
    return [c for c in df.columns if c not in ignore]


def train_models(train_df: pd.DataFrame, feature_cols, random_state: int = 42):
    groups = train_df["play_key"].astype(str).values
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
    train_idx, val_idx = next(splitter.split(train_df, groups=groups))

    tr = train_df.iloc[train_idx].reset_index(drop=True)
    va = train_df.iloc[val_idx].reset_index(drop=True)

    x_tr = tr[list(feature_cols)]
    x_va = va[list(feature_cols)]

    y_dx_tr = tr["target_dx"]
    y_dy_tr = tr["target_dy"]

    model_dx = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=random_state,
        n_jobs=-1,
    )

    model_dy = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=random_state,
        n_jobs=-1,
    )

    model_dx.fit(x_tr, y_dx_tr)
    model_dy.fit(x_tr, y_dy_tr)

    pred_dx = model_dx.predict(x_va)
    pred_dy = model_dy.predict(x_va)

    pred_x = va["last_x"].values + pred_dx
    pred_y = va["last_y"].values + pred_dy

    rmse_x = float(np.sqrt(mean_squared_error(va["target_x"], pred_x)))
    rmse_y = float(np.sqrt(mean_squared_error(va["target_y"], pred_y)))
    rmse_distance = float(np.sqrt(np.mean((pred_x - va["target_x"].values) ** 2 + (pred_y - va["target_y"].values) ** 2)))

    metrics = {
        "rmse_x": rmse_x,
        "rmse_y": rmse_y,
        "rmse_distance": rmse_distance,
        "n_train_rows": int(len(tr)),
        "n_val_rows": int(len(va)),
        "n_features": int(len(feature_cols)),
    }

    val_predictions = va[["game_id", "play_id", "nfl_id", "future_frame_id", "target_x", "target_y"]].copy()
    val_predictions["pred_x"] = pred_x
    val_predictions["pred_y"] = pred_y

    return model_dx, model_dy, metrics, val_predictions


def save_artifacts(model_dir: Path, model_dx, model_dy, feature_cols, last_k: int, metrics=None):
    model_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(model_dx, model_dir / "model_dx.joblib")
    joblib.dump(model_dy, model_dir / "model_dy.joblib")

    metadata = {
        "feature_columns": list(feature_cols),
        "last_k": int(last_k),
    }
    if metrics is not None:
        metadata["metrics"] = metrics

    with open(model_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)


def load_artifacts(model_dir: Path):
    model_dx = joblib.load(model_dir / "model_dx.joblib")
    model_dy = joblib.load(model_dir / "model_dy.joblib")

    with open(model_dir / "metadata.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)

    feature_cols = metadata["feature_columns"]
    last_k = int(metadata["last_k"])
    return model_dx, model_dy, feature_cols, last_k


def make_predictions(input_df: pd.DataFrame, model_dx, model_dy, feature_cols, last_k: int) -> pd.DataFrame:
    pred_table = build_prediction_table(input_df=input_df, last_k=last_k)

    x_pred = pred_table[list(feature_cols)]
    pred_dx = model_dx.predict(x_pred)
    pred_dy = model_dy.predict(x_pred)

    pred_table["x"] = pred_table["last_x"] + pred_dx
    pred_table["y"] = pred_table["last_y"] + pred_dy

    return pred_table[["game_id", "play_id", "nfl_id", "future_frame_id", "x", "y"]].rename(
        columns={"future_frame_id": "frame_id"}
    )

In [14]:
# =========================
# Load data
# =========================
train_input = pd.read_csv(TRAIN_INPUT_PATH)
train_output = pd.read_csv(TRAIN_OUTPUT_PATH)

check_columns(train_input, REQUIRED_INPUT_COLUMNS, "train_input")
check_columns(train_output, REQUIRED_OUTPUT_COLUMNS, "train_output")

print("train_input shape:", train_input.shape)
print("train_output shape:", train_output.shape)

train_input.head()

train_input shape: (285714, 25)
train_output shape: (32088, 9)


,game_id,play_id,player_to_predict,nfl_id,frame_id,play_direction,absolute_yardline_number,player_name,player_height,player_weight,...,y,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y,original_play_direction,was_flipped
0,2023090700,101,False,54527,1,right,42.0,Bryan Cook,6-1,210,...,36.94,0.09,0.39,322.40,238.24,21,63.259998,-0.22,right,False
1,2023090700,101,False,54527,2,right,42.0,Bryan Cook,6-1,210,...,36.94,0.04,0.61,200.89,236.05,21,63.259998,-0.22,right,False
2,2023090700,101,False,54527,3,right,42.0,Bryan Cook,6-1,210,...,36.93,0.12,0.73,147.55,240.60,21,63.259998,-0.22,right,False
3,2023090700,101,False,54527,4,right,42.0,Bryan Cook,6-1,210,...,36.92,0.23,0.81,131.40,244.25,21,63.259998,-0.22,right,False
4,2023090700,101,False,54527,5,right,42.0,Bryan Cook,6-1,210,...,36.90,0.35,0.82,123.26,244.25,21,63.259998,-0.22,right,False


In [15]:
# Optional quick checks
print("Number of plays in input:", train_input[["game_id", "play_id"]].drop_duplicates().shape[0])
print("Number of target-player rows in input:", (train_input["player_to_predict"] == 1).sum())
print("Output rows:", len(train_output))

Number of plays in input: 819
Number of target-player rows in input: 76399
Output rows: 32088


In [16]:
# =========================
# Build training table
# =========================
train_df = build_training_table(train_input, train_output, last_k=LAST_K)
feature_cols = get_feature_columns(train_df)

print("training table shape:", train_df.shape)
print("number of features:", len(feature_cols))

train_df.head()

training table shape: (32088, 68)
number of features: 59


,game_id,play_id,nfl_id,play_key,future_frame_id,horizon_t,horizon_frac,target_x,target_y,target_dx,...,hist_4_o_sin,hist_4_o_cos,hist_5_x_rel,hist_5_y_rel,hist_5_s,hist_5_a,hist_5_dir_sin,hist_5_dir_cos,hist_5_o_sin,hist_5_o_cos
0,2023090700,101,44930,2023090700_101,1,1,0.047619,53.20,13.98,0.77,...,0.97822,-0.20757,0.0,0.0,7.9,2.68,0.986996,-0.160743,0.957319,-0.289032
1,2023090700,101,44930,2023090700_101,2,2,0.095238,53.96,13.78,1.53,...,0.97822,-0.20757,0.0,0.0,7.9,2.68,0.986996,-0.160743,0.957319,-0.289032
2,2023090700,101,44930,2023090700_101,3,3,0.142857,54.70,13.54,2.27,...,0.97822,-0.20757,0.0,0.0,7.9,2.68,0.986996,-0.160743,0.957319,-0.289032
3,2023090700,101,44930,2023090700_101,4,4,0.190476,55.41,13.27,2.98,...,0.97822,-0.20757,0.0,0.0,7.9,2.68,0.986996,-0.160743,0.957319,-0.289032
4,2023090700,101,44930,2023090700_101,5,5,0.238095,56.09,12.95,3.66,...,0.97822,-0.20757,0.0,0.0,7.9,2.68,0.986996,-0.160743,0.957319,-0.289032


In [17]:
# =========================
# Train baseline models
# =========================
model_dx, model_dy, metrics, val_predictions = train_models(
    train_df=train_df,
    feature_cols=feature_cols,
    random_state=RANDOM_STATE,
)

print(json.dumps(metrics, indent=2))
val_predictions.head()

{
  "rmse_x": 1.0397797078817095,
  "rmse_y": 1.0987390453845725,
  "rmse_distance": 1.5127357108150696,
  "n_train_rows": 25944,
  "n_val_rows": 6144,
  "n_features": 59
}


,game_id,play_id,nfl_id,future_frame_id,target_x,target_y,pred_x,pred_y
0,2023090700,713,44888,1,41.71,40.683333,41.722553,40.698709
1,2023090700,713,44888,2,41.75,41.213333,41.746062,41.200238
2,2023090700,713,44888,3,41.76,41.763333,41.774585,41.664570
3,2023090700,713,44888,4,41.76,42.353333,41.804846,42.095670
4,2023090700,713,44888,5,41.75,42.953333,41.785465,42.541481


In [18]:
# =========================
# Save model artifacts
# =========================
save_artifacts(
    model_dir=MODEL_DIR,
    model_dx=model_dx,
    model_dy=model_dy,
    feature_cols=feature_cols,
    last_k=LAST_K,
    metrics=metrics,
)

val_predictions.to_csv(MODEL_DIR / "validation_predictions.csv", index=False)
pd.DataFrame([metrics]).to_csv(MODEL_DIR / "validation_metrics.csv", index=False)

print(f"Saved artifacts to: {MODEL_DIR.resolve()}")

Saved artifacts to: /content/drive/MyDrive/DataScience/baseline_model


In [19]:
# =========================
# Demo inference
# =========================
loaded_dx, loaded_dy, loaded_feature_cols, loaded_last_k = load_artifacts(MODEL_DIR)

predictions = make_predictions(
    input_df=train_input,
    model_dx=loaded_dx,
    model_dy=loaded_dy,
    feature_cols=loaded_feature_cols,
    last_k=loaded_last_k,
)

predictions.to_csv("baseline_predictions_demo.csv", index=False)

print("prediction shape:", predictions.shape)
predictions.head()

prediction shape: (32088, 6)


,game_id,play_id,nfl_id,frame_id,x,y
0,2023090700,101,44930,1,52.966495,13.759830
1,2023090700,101,44930,2,53.703547,13.636415
2,2023090700,101,44930,3,54.627698,13.490178
3,2023090700,101,44930,4,55.359419,13.183060
4,2023090700,101,44930,5,56.059198,12.894314
